# Phase 1: Validation Contract, 5-Fold Stratification & Diagnostic Profiler
## Amazon ML Challenge 2026 — Multilingual Business Entity Resolution

This notebook executes the full Phase 1 foundation:
1. **Step 1.0 (Validation Contract):** Exports canonical metric rules, seeds, and leakage boundaries.
2. **Step 1.1 (5-Fold Stratified Split):** Generates permanent `train_folds.parquet` stratified by country, match cardinality, and address presence.
3. **Step 1.2 (Diagnostic Evaluator):** Analyzes Candidate-Recall Curves at $K=5..150$, Oracle ceilings, and Loss Attribution.
4. **Step 1.3 (Ground Truth Profiler):** Measures exact signal coverage and multi-key blocking combinations across true pairs.
5. **Step 1.4 (French Robustness Benchmark):** Generates synthetic French morphological stress test.

In [ ]:
# Cell 1: Environment & Module Setup
import os
import sys
import time
import json
from pathlib import Path
import polars as pl
import numpy as np

# Locate project root
current_path = Path.cwd().resolve()
while current_path.parent != current_path:
    if (current_path / "dataset").exists() or (current_path / "code").exists():
        project_root = current_path
        break
    current_path = current_path.parent
else:
    project_root = Path.cwd().resolve()

ber_code_dir = project_root / "code" / "business_entity_resolution"
if str(ber_code_dir) not in sys.path:
    sys.path.insert(0, str(ber_code_dir))

print(f"Project Root: {project_root}")
print(f"BER Code Dir: {ber_code_dir}")

from src.config import PROCESSED_DIR, OUTPUT_DIR, TRAIN_DIR, RANDOM_SEED
from src.validation_contract import VALIDATION_CONTRACT, export_validation_contract, compute_entity_f05
from src.split_generator import generate_stratified_folds, load_fold_split
from src.evaluator import DiagnosticEvaluator
from src.profile_ground_truth import profile_ground_truth_signals
from src.french_robustness import generate_french_robustness_benchmark

print("All Phase 1 modules imported successfully!")

In [ ]:
# Cell 2: Step 1.0 — Export Validation Contract
contract_file = PROCESSED_DIR / "validation_contract.json"
export_validation_contract(contract_file)

with open(contract_file, "r", encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# Cell 3: Step 1.1 — Generate 5-Fold Stratified Mapping
folds_df = generate_stratified_folds(
    s1_path=PROCESSED_DIR / "train_source1_cleaned.parquet",
    output_path=PROCESSED_DIR / "train_folds.parquet",
    num_folds=5,
    random_state=42
)

print("\nSample Fold Records:")
print(folds_df.head(10))

In [ ]:
# Cell 4: Step 1.3 — Run Ground Truth Signal & Combination Profiler
gt_profile = profile_ground_truth_signals(
    sample_size=50000,
    output_json=PROCESSED_DIR / "gt_signal_profile.json"
)

In [ ]:
# Cell 5: Step 1.4 — Generate French Linguistic Robustness Benchmark
fr_robustness_df = generate_french_robustness_benchmark(
    num_samples=15000,
    output_path=PROCESSED_DIR / "val_synthetic_france.parquet"
)

print("\nSample French Robustness Records:")
print(fr_robustness_df.head(5))

In [ ]:
# Cell 6: Step 1.2 — Test Diagnostic Evaluator & Ceiling Analyzer
evaluator = DiagnosticEvaluator()

# Example check with mock or validation predictions
print(f"Evaluator active with {len(evaluator.ground_truth):,} true ground truth entities.")